In [18]:
from omegaconf import OmegaConf
from jupyterscad import view
from itertools import accumulate

from solid2 import square

from configuration import schema, KeyDimensions, ConfigSchema

In [19]:
yaml_config = OmegaConf.load("default.conf.yaml")
conf = OmegaConf.merge(schema, yaml_config)
conf

{'dist_u': 19.05, 'white_key_dims': {'length': 1.75, 'width': 1.25}, 'black_key_dims': {'length': 1.75, 'width': 1.0}, 'rows_height_diff_mm': 12.0, 'mount_plate_width': 1.25, 'mount_u': 14.0, 'keycap_u': 18.0, 'keycap_height_mm': 1.5, 'keycap_rounding_corner_mm': 2.0, 'output_dir': PosixPath('build')}

In [ ]:
def generate_keys_row(
    plate_width: float,
    plate_length: float,
    key_sep_distances: list[float],
    y_offset: float,
    conf: ConfigSchema,
):
    u = conf.mount_u
    mx_hole = square([u, u]).translateY(y_offset)
    mounting_plate = square([plate_width, plate_length])

    for sep in accumulate(key_sep_distances):
        mounting_plate -= mx_hole.translateX(sep)

    return mounting_plate.linear_extrude(conf.mount_plate_width)

In [49]:
wk_total_width = conf.white_key_dims.width * conf.dist_u
single_key_len = conf.white_key_dims.length 
white_plate_width = wk_total_width * 7
white_plate_len = single_key_len * conf.dist_u
distances = [
    (wk_total_width - conf.mount_u) / 2
] + [wk_total_width] * 6
white_mount_plate = generate_keys_row(
    white_plate_width,
    white_plate_len,
    distances,
    (white_plate_len - conf.mount_u) / 2,
    conf,
)

view(white_mount_plate)

4.90625
28.71875
52.53125
76.34375
100.15625
123.96875
147.78125


Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.7, position=(3.0, 5.0,…

In [ ]:
bk_width_total = conf.black_key_dims.width * conf.dist_u
single_key_len = conf.black_key_dims.length 
white_plate_width = bk_width_total * 7
white_plate_len = conf.dist_u
distances = [
    ((bk_width_total - conf.mount_u) / 2),
    wk_total_width,
    wk_total_width * 2,
    wk_total_width,
    wk_total_width,

] 
black_mount_plate = generate_keys_row(
    white_plate_width,
    white_plate_len,
    distances,
    (white_plate_len - conf.mount_u) / 2,
    conf,
)

view(black_mount_plate + white_mount_plate.translateY(white_plate_len))

26.3375
50.15
97.775
121.5875
145.4


Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.7, position=(3.0, 5.0,…

In [ ]:
# TODO: each octave is indivisible part 
#       but it is possible to generate half of the octave
#       currently I have 35 switches, so the max I can get is 2.5 octaves.
#       30~ keys + 5 mods (octave up, octave down, maybe some play, record, etc)


